# Lecture 4 Activity – Did the South change parties? (solutions)

QSS 20 · Modern Statistical Computing · Fall 2026

## The question

In lecture we found that the two parties in the House moved apart between 1947
and 2013, that both moved, and that by 2011 they did not overlap at all.

That leaves a question the lecture could not answer. **Where did the movement
come from?**

One long-standing account in American politics is the **Southern
realignment**. The South was solidly Democratic in the 1940s and solidly
Republican by the 2010s.

If that is right, we should see it in the same table, using only selection.

> **Did the South switch parties between the 80th Congress and the 112th, and
> what happened to the Southern Democrats who stayed?**

### Reminders

- One row is **one member of the House in one Congress**.
- `district == 0` rows are **presidents**, not House members.
- `dwnom1`: negative liberal, positive conservative.
- `and` collapses a column into one answer; `&` keeps one answer per row. You
  want `&`, and each condition needs its own parentheses.

In [1]:
import pandas as pd

pd.set_option('display.max_rows', 8)

congress = pd.read_csv('../data/raw/congress.csv')
congress.head(3)

,congress,district,state,party,name,dwnom1,dwnom2
0,80,0,USA,Democrat,TRUMAN,-0.276,0.016
1,80,1,ALABAMA,Democrat,BOYKIN F.,-0.026,0.796
2,80,2,ALABAMA,Democrat,GRANT G.,-0.042,0.999


## Task 0 · Drop the presidents

Build `house`, the table with the president rows removed, and print how many
rows you dropped.

<details><summary>Hint</summary>

`!=` means "not equal to". You want rows where `district` is not 0.

</details>

In [2]:
house = congress.loc[congress['district'] != 0]
print('dropped:', congress.shape[0] - house.shape[0], 'president rows')
print('house rows:', house.shape[0])

dropped: 35 president rows
house rows: 14517


## Task 1 · Which states are in the file, exactly?

You are about to select Southern states by name. Before you write the list,
print the unique values of the `state` column.

Then answer in the Markdown cell below: what do you notice, and what does it
mean for the list you were about to write?

In [3]:
house['state'].unique()

array(['ALABAMA', 'ARIZONA', 'ARKANSA', 'CALIFOR', 'COLORAD', 'CONNECT',
       'DELAWAR', 'FLORIDA', 'GEORGIA', 'IDAHO', 'ILLINOI', 'INDIANA',
       'IOWA', 'KANSAS', 'KENTUCK', 'LOUISIA', 'MAINE', 'MARYLAN',
       'MASSACH', 'MICHIGA', 'MINNESO', 'MISSISS', 'MISSOUR', 'MONTANA',
       'NEBRASK', 'NEVADA', 'NEW HAM', 'NEW JER', 'NEW MEX', 'NEW YOR',
       'NORTH C', 'NORTH D', 'OHIO', 'OKLAHOM', 'OREGON', 'PENNSYL',
       'RHODE I', 'SOUTH C', 'SOUTH D', 'TENNESS', 'TEXAS', 'UTAH',
       'VERMONT', 'VIRGINI', 'WASHING', 'WEST VI', 'WISCONS', 'WYOMING',
       'ALASKA', 'HAWAII'], dtype=object)

**Answer.** The values are cut to seven characters, so `'LOUISIANA'` and
`'MISSISSIPPI'` match nothing at all. Any list has to use the truncated
spellings.

A name that matches nothing raises no error. It silently contributes no rows,
and your count comes out too small.

## Task 2 · Build the Southern delegation

Using `.isin()`, select members from these eleven states, in the **80th
Congress** only:

Alabama, Arkansas, Florida, Georgia, Louisiana, Mississippi, North Carolina,
South Carolina, Tennessee, Texas, Virginia.

Use the spellings you found in Task 1. Print the row count, and check that all
eleven states actually appear.

In [4]:
south = ['ALABAMA', 'ARKANSA', 'FLORIDA', 'GEORGIA', 'LOUISIA', 'MISSISS',
         'NORTH C', 'SOUTH C', 'TENNESS', 'TEXAS', 'VIRGINI']

south80 = house.loc[(house['congress'] == 80) & (house['state'].isin(south))]
print('rows:', south80.shape[0])
print('states matched:', south80['state'].nunique(), 'of', len(south))

rows: 107
states matched: 11 of 11


**Answer.** 107 members, and all eleven states matched. If you see fewer than eleven,
one of your strings is wrong; `.nunique()` on the selection is the fastest way
to find that out.

## Task 3 · ❓ Before you count

You are about to split the Southern delegation you just built into Democrats
and Republicans, for 1947.

**Write your guess down first.** Out of those members, what share were
Republicans: about a tenth, about a quarter, about half?

Then compute it, and do the same for the 112th Congress.

In [5]:
for c in [80, 112]:
    s = house.loc[(house['congress'] == c) & (house['state'].isin(south))]
    print(f"Congress {c}: {s.shape[0]} members")
    print(s['party'].value_counts().to_string(), end='\n\n')

Congress 80: 107 members
party
Democrat      105
Republican      2

Congress 112: 131 members
party
Republican    94
Democrat      37



**Answer.** 80th: 107 members, **105 Democrats and 2 Republicans**.
112th: 131 members, **37 Democrats and 94 Republicans**.

Most people guess somewhere between 10 and 30 Republicans for 1947. The real
answer is 2. The South went from 98 percent Democratic to 72 percent
Republican, which is about as complete a party switch as a region can
produce.

## Task 4 · How did it happen?

A region can switch parties two ways, and political scientists have names for
them:

- **Conversion**: the same people keep their seats and change how they vote.
- **Replacement**: the old members leave, and people who vote differently take
  their seats.

Party labels alone cannot tell the two apart, because either one turns a
Democratic seat into a Republican one. The scores can, and so can the counts.

Compare the mean `dwnom1` of **Southern Democrats** with **non-Southern
Democrats**, in the 80th and again in the 112th. Report the group sizes too.

You will need `~`, which flips a condition.

In [6]:
for c in [80, 112]:
    sd = house.loc[(house['congress'] == c) & (house['state'].isin(south))
                   & (house['party'] == 'Democrat'), 'dwnom1']
    nd = house.loc[(house['congress'] == c) & (~house['state'].isin(south))
                   & (house['party'] == 'Democrat'), 'dwnom1']
    print(f"Congress {c}: South n={len(sd):3d} mean={sd.mean():+.3f} | "
          f"elsewhere n={len(nd):3d} mean={nd.mean():+.3f} | gap {sd.mean() - nd.mean():+.3f}")

Congress 80: South n=105 mean=-0.027 | elsewhere n= 88 mean=-0.285 | gap +0.258
Congress 112: South n= 37 mean=-0.342 | elsewhere n=162 mean=-0.406 | gap +0.064


**Answer.** 80th: Southern Democrats averaged -0.027 against -0.285 elsewhere, a gap of
0.26. They were far more conservative than the rest of their own party.

112th: -0.342 against -0.406, a gap of 0.06. The two groups now vote almost
alike.

So the Southern Democrats who are left look like ordinary Democrats. And the
counts show there are far fewer of them: 105 in the 80th, 37 in the 112th.

Careful here. **Both** facts are exactly what conversion predicts, and both are
exactly what replacement predicts. A group mean can move because its members
moved, or because different members are in it. Task 5 is how we tell.

## Task 5 · ❓ Which was it?

To tell conversion from replacement you have to follow the **same person**
across Congresses, and see whether their own score moved.

George Mahon of Texas held the 19th district seat for decades. Select his rows
and print `congress` and `dwnom1` for every Congress he served.

Before you run it, write down a guess: over roughly 27 years, how far do you
think his score moved?

In [7]:
house.loc[(house['state'] == 'TEXAS') & (house['district'] == 19)
          & (house['name'] == 'MAHON'), ['congress', 'dwnom1']]

,congress,dwnom1
1291,82,-0.088
1724,83,-0.089
2162,84,-0.091
2605,85,-0.093
...,...,...
4811,90,-0.103
5254,91,-0.105
6576,94,-0.110
7015,95,-0.112


**Answer.** His score goes from **-0.088 to -0.112** across twelve Congresses, about 27
years. It moved **0.024**, and it crept there in tiny, almost perfectly even
steps.

Now compare. The Southern Democratic group mean moved from -0.027 to -0.342,
a shift of **0.315**, roughly **thirteen times** as far as Mahon moved in his
whole career.

Most members look like Mahon. DW-NOMINATE gives each member a nearly fixed
position that can drift only slowly, so a member who genuinely changed sides
would barely register. If individual scores hardly move but the group mean
moves a great deal, the movement has to come from **who is in the group**.

That is the argument for replacement: not that conversion is impossible, but
that it is far too small here to produce a shift of 0.315.

## Task 6 · State the boundary

In the Markdown cell below, write two or three sentences answering:

1. What do these numbers support saying about the Southern realignment?
2. Name one thing they cannot establish. Think about what one row is, and about
   what `dwnom1` is built from.

In [8]:
# No code for this task.

**Answer.** A defensible answer: the Southern delegation switched party label, the
surviving Southern Democrats now vote like Democrats elsewhere, and because
individual scores barely move, most of the change has to be replacement rather
than conversion.

What it cannot establish. One member is not a proof, and checking every member
is harder than it looks: surnames are not unique identifiers. The Texas 20th is
held by `GONZALEZ` for eighteen Congresses in this table, but that is a father
and then his son, scored as if they were one person. So this table can point at
replacement without cleanly measuring it.

It also cannot say **why** anyone switched. One row is a member in a Congress,
and nothing here records a voter, a district's composition, or anyone's
reasons. `dwnom1` is built from roll-call agreement, so it cannot separate a
changed belief from a changed party line.

## What to take from this

- A condition that matches nothing raises no error. Check a column's values
  with `.unique()` before you filter on it, and check the row count after.
- A count and a mean answer different questions. Here the count carried the
  story: Southern Democrats fell from 105 to 37.
- A group mean can move because its members moved, or because its membership
  changed. Comparing one member's own movement against the group's is how you
  tell the two apart.